Paquetes necesarios

In [ ]:
import cv2
import time
import numpy as np

**TAREA**: Los ejemplos ilustrativos anteriores permiten saber el número de monedas presentes en la imagen ideal proporcionada. ¿Cómo determinar la cantidad de dinero presente en ella? A partir únicamente de las dimensiones, una sugerencia implicaría, en primer término, identificar de forma interactiva (por ejemplo haciendo clic en la imagen) una moneda de un valor fijo en la imagen (por ejemplo de 1€) para permitir establecer una relación entre píxeles y milímetros. Tras obtener esa información y las dimensiones en milímetros de las distintas monedas, es factible calcular la cantidad de dinero en la imagen ideal.

Una vez resuelto el reto con la imagen ideal proporcionada, diseña un demostrador en vivo que a partir de la entrada de vídeo de la cámara, determine el dinero presente en la escena. Si bien no hay restricciones sobre utilizar medidas geométricas o de color, o cualquier clasificador, no deben hacer uso de modelos de aprendizaje profundo. ¿Funciona correctamente? ¿Se observan problemas?
Posibles extras serían la presencia de objetos que no son monedas y/o haya solape entre las monedas.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo

Nota: Para establecer la correspondencia entre píxeles y milímetros, comentar que la moneda de un euro tiene un diámetro de 23.25 mm. la de 50 céntimos de 24.35, la de 20 céntimos de 22.25, etc. 



In [ ]:
vid = cv2.VideoCapture(1)

reference_objet_size = 27
pixel_size_buffer = []
pixel_size = 0 # mm/pixel

calibration_seconds = 5
start_time = time.time()

money_amount = {8.125: 0.01, 9.375: 0.02, 9.875: 0.1, 10.625: 0.05, 11.125: 0.2, 11.625: 1, 12.125: 0.5, 12.875: 2}
amounts_readed = []

umbral = 0.25

def calculate_radius(contours, binary_image, distance_image, calibrating):
    total_amount = 0
    for index, contour in enumerate(contours):
        old_difference = float('inf')
        # Eliminamos ruido de contornos pequeños o demasiado grandes
        if cv2.contourArea(contour) < 100 or cv2.contourArea(contour) > 10000:
            continue
        # Creamos una máscara para el contorno actual y calculamos el valor máximo de la transformada de distancia dentro de esa máscara
        mask = np.zeros(binary_image.shape, dtype=np.uint8)
        cv2.drawContours(mask, contours, index, 255, thickness=-1)

        _, max_val, _, _ = cv2.minMaxLoc(distance_image, mask=mask)

        if calibrating:
            # Calculamos el tamaño de pixel a partir del radio de referencia y lo añadimos al buffer
            pixel_size_buffer.append(reference_objet_size / max_val)
        else:
            # Calculamos el radio en mm a partir del valor máximo de la transformada de distancia y el tamaño de pixel
            radius = max_val * pixel_size
            for key in money_amount.keys():
                # Calculamos la diferencia entre el radio calculado y el radio de la moneda correspondiente
                difference = abs(radius - key)
                if difference <= umbral:
                    print(max_val)
                    print(f'Radio: {radius:.3f} mm, Valor: {money_amount[key]}')
                    # Sumamos el valor de la moneda al total si la diferencia es menor o igual al umbral
                    total_amount += money_amount[key]
                    break
                # Si la diferencia deja de disminuir, significa que no se encontró un valor correspondiente y se rompe el bucle
                if difference <= old_difference:
                    old_difference = difference
                else:
                    print(f'Escaneo una moneda con radio: {radius:3f} mm, pero no se encontró un valor correspondiente. Diferencia mínima: {old_difference:.3f} mm')
                    break
    if not calibrating:
        # Añadimos el total de la mesa a la lista de lecturas
        amounts_readed.append(total_amount)
        # Se muestra el valor de la mesa como la mediana de las lecturas obtenidas
        print(f'Valor en la mesa: {np.median(amounts_readed)}')

while(True):
    # Leemos la entrada de la cámara fotograma a fotograma
    ret, frame = vid.read()

    if ret:
        # Convertimos a escala de grises y binarizamos la imagen
        grey_image = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        _, binarized_image = cv2.threshold(grey_image, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

        # Calculamos la transformada de distancia y los contornos de la imagen binarizada
        img_dist = cv2.distanceTransform(binarized_image, cv2.DIST_L2, 5)
        contours, _ = cv2.findContours(binarized_image, cv2.RETR_EXTERNAL , cv2.CHAIN_APPROX_SIMPLE)

        # Dibujamos los contornos sobre la imagen original para mostrar al usuario
        rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        cv2.drawContours(rgb_image, contours, -1, (0,255,0), 3)
        bgr_frame = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)

        # Calibramos el fondo durante los primeros segundos
        if time.time() - start_time < calibration_seconds:
            # Leemos el radio del contorno de referencia y establecemos un tamaño de pixel
            calculate_radius(contours, binarized_image, img_dist, True)

            # Calculamos el tamaño de pixel a partir de la mediana de las lecturas previas
            pixel_size = np.median(pixel_size_buffer)

            # Mostramos los fotogramas para aportar feedback visual al usuario
            cv2.imshow('Fotograma', bgr_frame)
            cv2.imshow('Transformada de Distancia', img_dist)

            # Detenemos pulsando ESC
            if cv2.waitKey(20) == 27:
                break

            continue

        calculate_radius(contours, binarized_image, img_dist, False)
        # Mostramos los fotogramas para aportar feedback visual al usuario
        cv2.imshow('Fotograma', bgr_frame)
        cv2.imshow('Transformada de Distancia', img_dist)

    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

# Liberamos el objeto de captura y destruimos las ventanas
vid.release()
cv2.destroyAllWindows()

**TAREA**: La tarea consiste en extraer características (geométricas y/o visuales) de las tres imágenes completas de partida, y *aprender* patrones que permitan identificar las partículas en nuevas imágenes. Para ello se proporciona como imagen de test *MPs_test.jpg* y sus correpondientes anotaciones *MPs_test_bbs.csv* con la que deben obtener las métricas para su propuesta de clasificación de microplásticos, además de la matriz de confusión. La matriz de confusión permitirá mostrar para cada clase el número de muestras que se clasifican correctamente de dicha clase, y el número de muestras que se clasifican incorrectamente como perteneciente a una de las otras dos clases. De nuevo, no deben hacer uso de modelos de aprendizaje profundo ni para segmentar, ni para clasificar.

En el trabajo [SMACC: A System for Microplastics Automatic Counting and Classification](https://doi.org/10.1109/ACCESS.2020.2970498), las características geométricas utilizadas fueron:

- Área en píxeles
- Perímetro en píxeles
- Compacidad (relación entre el cuadrado del perímetro y el área de la partícula)
- Relación del área de la partícula con la del contenedor
- Relación del ancho y el alto del contenedor
- Relación entre los ejes de la elipse ajustada
- Definido el centroide, relación entre las distancias menor y mayor al contorno

Si no se quedan satisfechos con la segmentación obtenida, es el mundo real, también en el README comento técnicas recientes de segmentación, que podrían despertar su curiosidad.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo